# Notebook 1: Data Preparation

**Flood Susceptibility Mapping — Karabük, Turkey**
CME434, Karabük University

**Purpose:** Load GEE training CSV, clean it, and save `Inputs.txt` + `Label.txt`
**Input:**  `data/raw/Training_Dataset.csv` (downloaded from Google Drive)
**Output:** `data/processed/Inputs.txt`, `data/processed/Label.txt`

## Step 1 — Import Libraries

In [ ]:
# Run this cell first
# In Google Colab this works out of the box
# Locally: pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("Libraries loaded")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

## Step 2 — Load Training CSV from GEE

In [ ]:
# TODO: Download Training_Dataset.csv from Google Drive folder GIS_Flood_Karabuk
# Place in: data/raw/Training_Dataset.csv

# Colab — mount Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# csv_path = '/content/drive/MyDrive/GIS_Flood_Karabuk/Training_Dataset.csv'

csv_path = '../data/raw/Training_Dataset.csv'
df = pd.read_csv(csv_path)
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(df.head(3))

## Step 3 — Drop GEE Metadata Columns

In [ ]:
drop_list = ['system:index', '.geo', 'latitude', 'longitude',
             'longitude_1', 'latitude_1', 'geo']
cols_to_drop = [c for c in df.columns
                if c in drop_list or c.startswith('system')]
print(f"Dropping: {cols_to_drop}")
df_clean = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Remaining columns: {df_clean.columns.tolist()}")
print(f"Shape: {df_clean.shape}")

## Step 4 — Handle Missing Values

In [ ]:
print("Missing values per column:")
missing = df_clean.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")
print(f"Total missing cells: {df_clean.isnull().sum().sum()}")
df_clean = df_clean.dropna()
print(f"Shape after dropna: {df_clean.shape}")

## Step 5 — Separate Features (X) and Labels (y)

In [ ]:
label_col = 'Label'  # 1 = flooded, 0 = non-flooded

if label_col not in df_clean.columns:
    print(f"ERROR: column '{label_col}' not found")
    print(f"Available: {df_clean.columns.tolist()}")
else:
    X = df_clean.drop(columns=[label_col])
    y = df_clean[label_col]
    print(f"X shape: {X.shape}  |  y shape: {y.shape}")
    print(f"\nClass distribution:\n{y.value_counts()}")
    print(f"\n{y.mean()*100:.1f}% flooded points")

## Step 6 — Feature Distributions

In [ ]:
print(X.describe().round(3))

os.makedirs('../outputs/figures', exist_ok=True)
n = X.shape[1]
ncols, nrows = 5, (n + 4) // 5
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*2.5))
axes = axes.flatten()

for i, col in enumerate(X.columns):
    axes[i].hist(X[col].dropna(), bins=30, color='steelblue',
                 edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions — Flood Susceptibility', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/figures/feature_distributions.png")

## Step 7 — Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 10))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, mask=mask,
            annot_kws={'size': 7}, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/figures/correlation_matrix.png")

print("\nHighly correlated pairs (|r| > 0.8):")
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8:
            print(f"  {corr.columns[i]} / {corr.columns[j]}: "
                  f"{corr.iloc[i,j]:.2f}")

## Step 8 — Save Inputs.txt and Label.txt

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
np.savetxt('../data/processed/Inputs.txt', X.values)
np.savetxt('../data/processed/Label.txt',  y.values)

print("Saved:")
print(f"  data/processed/Inputs.txt  shape={X.shape}")
print(f"  data/processed/Label.txt   shape={y.shape}")
print("\nFeature column order (= column order in Inputs.txt):")
for i, col in enumerate(X.columns):
    print(f"  col {i:2d}: {col}")
print("\nDone. Next: colab/02_model_training.ipynb")